# EuroCropML Benchmark - Step 2: Data Preparation
Extract classical features in chunks and save to Google Drive.
Processes 140K samples without exceeding 12GB RAM.

In [ ]:
%cd eurocrop-olmoearth-benchmark
import numpy as np
import os

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/eurocrop_benchmark'

In [ ]:
# NDVI features: (T, C) -> (4,)
def ndvi_single(x):
    B4, B8 = 3, 7
    red = x[:, B4].astype(np.float32)
    nir = x[:, B8].astype(np.float32)
    ndvi = (nir - red) / (nir + red + 1e-8)
    return np.array([ndvi.mean(), ndvi.max(), ndvi.min(), ndvi.std()], dtype=np.float32)

# Band stat features: (T, C) -> (39,)
def band_stat_single(x):
    x32 = x.astype(np.float32)
    return np.concatenate([x32.mean(0), x32.std(0), x32.max(0)]).astype(np.float32)

In [ ]:
# Extract NDVI features (chunked, ~2 min)
from src.data.prep_features import extract_features_chunked

ndvi_dir = os.path.join(DRIVE_DIR, 'features_ndvi')
extract_features_chunked(
    preprocess_dir='./preprocess',
    split_dir='./split',
    use_case='latvia_vs_estonia',
    feature_fn=ndvi_single,
    output_dir=ndvi_dir,
    chunk_size=5000
)

In [ ]:
# Extract band stat features (chunked, ~2 min)
bandstat_dir = os.path.join(DRIVE_DIR, 'features_bandstat')
extract_features_chunked(
    preprocess_dir='./preprocess',
    split_dir='./split',
    use_case='latvia_vs_estonia',
    feature_fn=band_stat_single,
    output_dir=bandstat_dir,
    chunk_size=5000
)

In [ ]:
# Verify saved features
from src.data.prep_features import load_prepared_features

for name, d in [("NDVI", ndvi_dir), ("BandStat", bandstat_dir)]:
    X_tr, y_tr, X_te, y_te, labels = load_prepared_features(d)
    print(f"{name}: train={X_tr.shape}, test={X_te.shape}, classes={len(labels)}")